# 7. From ROOT events to a reproducible fit

**Learning goals:** write a toy TTree, map branches into a phase-space sample, validate the round trip, and fit directly from ROOT.

Run cells from top to bottom in a fresh Python kernel. Install the package and Jupyter
as explained in [the course guide](TUTORIALS.md). No external data files are needed.
Masses are in GeV, invariants in GeV², and daughter indices start at zero.
The small event counts and grid sizes keep this lesson practical on a CPU; they are
teaching settings, not a demonstrated precision choice for a physics analysis.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=100, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

In [3]:
data = generate_toy(
    model, 2500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=384, include_momenta=False,
)
print(f"Generated {data.size} unweighted events")
assert np.all(np.asarray(data.weights) == 1)

## Create a local input file

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
from dalitzplotfitter import write_phase_space_sample, read_phase_space_sample

with TemporaryDirectory(prefix="dalitz-tutorial-") as temporary:
    path = Path(temporary) / "signal.root"
    write_phase_space_sample(path, data, tree="DecayTree", include_momenta=False)
    loaded = read_phase_space_sample(
        path, "DecayTree", s12="s12", s13="s13", s23="s23",
    )
    for name in ("s12", "s13", "s23"):
        np.testing.assert_array_equal(getattr(loaded, name), getattr(data, name))
    assert loaded.size == data.size
    print("Round trip verified:", loaded.size, "events")

    # Replace these branch names when reading a real experiment's TTree.
    session = FitSession.from_root(
        model, path, "DecayTree", s12="s12", s13="s13", s23="s23",
    )
    result = session.fit({"NR.x": 0.35, "NR.y": 0.45}, simplex=True, ncall=5000)
    assert result.valid
    session.report(result)

## Inspect and preserve the fit result

The session holds the loaded arrays, so it remains usable after closing the ROOT file.
Real event selections must be reflected in signal and background normalization when they
affect Dalitz acceptance. Without a weight branch, the reader supplies unit event weights;
loading weights alone does not define a statistically appropriate weighted likelihood.
These lessons use ordinary unweighted data throughout.

In [5]:
session.plot_projection(result, "s12", bins=40, projection_size=20000)
plt.show()
summary = {
    "valid": bool(result.valid),
    "nll": float(result.fval),
    "values": {name: float(result.values[name]) for name in result.parameters},
    "errors": {name: float(result.errors[name]) for name in result.parameters},
    "toy_seed": 2026,
    "normalization_resolution": 100,
}
import json
print(json.dumps(summary, indent=2))

## Paper-model tutorial section

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

## Try it yourself

1. Read only an entry range with `entry_start` and `entry_stop`.
2. Write a second tree using `write_phase_space_samples`.
3. Adapt the explicit branch mapping to your data, checking units and invariant identities before fitting.

## Continue learning

Reference: [ROOT I/O](../../docs/root_io.md). Continue with the advanced topics in [the course guide](TUTORIALS.md).

Return to [the course guide](TUTORIALS.md).